# None を返すのではなく例外を送出する

ユーティリティ関数を書くときに、Python プログラマは、None という戻り値に特別な意味を与えようとすることがあります。それがもっともだと思える場合があります。

例えば、ある数を別の数で割るヘルパー関数を書くとしましょう。ゼロで割る場合には、結果が未定義なので、None を返すのが自然に思えます。

In [8]:
def careful_divide(a, b):
  try:
    return a / b
  except ZeroDivisionError:
    return None

この関数を使うコードでは、戻り値をそれに従って解釈します。

In [2]:
x, y = 1, 0
result = careful_divide(x, y)
if result is None:
  print('Invalid inputs')

Invalid inputs


問題は、if 文のような条件で結果を評価しようとしていると、戻り値のゼロが面倒な事態を引き起こすことです。None だけに注目するのではなく、エラーを引き起こす False と判定される値にも注目する場合です。

In [ ]:
x, y = 0, 5
result = careful_divide(x, y)
print(result) # 0.0 だが Invalid inputs になってしまう
if not result:
  print('Invalid inputs') # これは動作するが、こうすべきではない

0.0
Invalid inputs


この False と等価な戻り値の誤った解釈は、None が特別な意味を持つときに、Python コードでよくある間違いです。careful_divide のように関数から None を返すことがエラーにつながりやすい理由がしめされています。このようなエラーを減らすには2つの方式があります。

第1の方法は、戻り値を2個のタプルにするものです。タプルの第1項は演算が成功したか失敗したかを示します。第2項が計算された実際の結果です。

In [10]:
def careful_divide(a, b):
  try:
    return True, a / b
  except ZeroDivisionError:
    return False, None

この関数の呼び出し元は、タプルをアンパックしなければなりません。それによって、除算の結果だけを見るのではなくて、タプルの状態部分を考慮させるようにします。

In [12]:
success, result = careful_divide(x, y)
if not success:
  print('Invalid inputs')
else:
  print(result)

0.0


問題は、呼び出し元がタプルの最初の部分を簡単に無視できることです。そのコードは一見問題なさそうですが、None を返すだけの場合と同様によくありません。

In [13]:
_, result = careful_divide(x, y)
if not result:
  print('Invalid inputs')

Invalid inputs


このようなエラーを減らす、もっと優れた第2の方法は、特別な場合に None をそもそも返さないことです。その代わりに、例外を呼び出し元に送出して、その処理を行わせます。次のコードでは、ZeroDivisionError を ValueError に変換して、呼び出し元に入力値が正しくないことを示します。

In [16]:
def careful_divide(a, b):
  try:
    return a / b
  except ZeroDivisionError:
    raise ValueError('Invalid inputs')

呼び出し元では、もはや関数の戻り値について調べる必要がありません。代わりに戻り値は常に政党だと仮定し、try の else ブロックで結果をすぐに使います。

In [15]:
x, y = 5, 2
try:
  result = careful_divide(x, y)
except ValueError:
  print('Invalid Inputs')
else:
  print(f'Result is {result:.1f}')

Result is 2.5


この方式は、型ヒントを使ったコードにも拡張できます。関数の戻り値は、常に float であり、None には絶対ならないように指定できます。

```Python
def sqrt(x: float) -> float:
    ...
```

と書けば「戻り値は float で None ではない」と型で示せる。

しかし、Python での緩やかな型付けは、そもそも例外が関数のインタフェースの一部であること（チェック例外と呼ばれる）を示す方法を意図的に提供していません。

Java だと以下のように「チェック例外」という仕組みがあります。

```Java
double sqrt(double x) throws NegativeValueException
```

と書くと、
- この関数は NegativeValueException を投げる可能性がある
- 呼び出し側は必ず try/catch で処理をしなければならない

ということが 型レベルで強制される。

しかし Python には、この “throws” のような機能がない。

Python の型ヒントでは、

- この関数がどんな例外を投げるのか
- 呼び出し側がその例外を捕捉すべきか

といった情報を 型ヒントとして書けない。

つまり、関数のインターフェース（入力と出力）に「例外を投げる」という情報を組み込む仕組みがない。

その代わりに、例外を送出する振る舞いをドキュメント化する必要があり、呼び出し元は、そのドキュメントを信頼して、どの例外を補足すべきかをあらかじめわかって計画するものと期待します（項目84）。

まとめると、型ヒントと docstring を使うと、この関数は次のように記述できます。

In [17]:
def careful_divide(a, b):
  """Divides a by b

  Raise:
     ValueError: When the inputs cannot be divided.
  """
  try:
    return a / b
  except ZeroDivisionError:
    raise ValueError('Invalid inputs')

これにより、入力、出力、例外の振る舞いが明確になり、呼び出し元が間違える回数がとても減ります。

## 覚えておくこと

- None を返して、特別な意味を示す関数は、None と（例えば、ゼロ、空文字列など）他の値とがすべて条件式において False に評価されるので、エラーを引き起こしやすい。
- None を返す代わりに、例外を送出して、特別な条件を示すようにする。呼び出し元のコードで、その処理が文書化されており、適切に例外処理することを期待する。
- 型ヒントを使って、関数が特別な場合にも絶対に None を返さないことを明示できる。